In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

# Phase 1b - Bronze Profiling

Profiles all 6 bronze tables (duplicates, blanks, text formatting, dates, numeric values, referential integrity), records every finding in the Data Quality Log, and contains the Phase 2 dimensional design.
**Not part of the pipeline:** this notebook is exploratory analysis and documentation that only reads bronze, so it doesn't load any table and doesn't need to run on every load.

In [0]:
-- Profiling step 0: bronze row counts vs the row counts stated in the README
SELECT 'customers'        AS table_name, COUNT(*) AS bronze_rows, 508   AS expected_rows FROM bronze_efkleia_urbanmart.customers
UNION ALL SELECT 'customer_changes', COUNT(*), 25    FROM bronze_efkleia_urbanmart.customer_changes
UNION ALL SELECT 'products',         COUNT(*), 76    FROM bronze_efkleia_urbanmart.products
UNION ALL SELECT 'stores',           COUNT(*), 13    FROM bronze_efkleia_urbanmart.stores
UNION ALL SELECT 'orders',           COUNT(*), 8000  FROM bronze_efkleia_urbanmart.orders
UNION ALL SELECT 'order_items',      COUNT(*), 24111 FROM bronze_efkleia_urbanmart.order_items;

table_name,bronze_rows,expected_rows
customers,508,508
customer_changes,25,25
products,76,76
stores,13,13
orders,8000,8000
order_items,24111,24111


**Profiling for customers table**

In [0]:
-- Same customer_id more than once? Exact copy or conflicting versions?
SELECT customer_id,
       COUNT(*)                         AS row_count,
       COUNT(DISTINCT _source_row_hash) AS distinct_versions
FROM bronze_efkleia_urbanmart.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id;

customer_id,row_count,distinct_versions
C0021,2,1
C0079,2,1
C0080,2,1
C0166,2,1
C0239,2,1
C0364,2,1
C0476,2,1
C0480,2,1


In [0]:
SELECT
  COUNT_IF(customer_id IS NULL OR trim(customer_id) = '') AS blank_customer_id,
  COUNT_IF(first_name  IS NULL OR trim(first_name)  = '') AS blank_first_name,
  COUNT_IF(last_name   IS NULL OR trim(last_name)   = '') AS blank_last_name,
  COUNT_IF(email       IS NULL OR trim(email)       = '') AS blank_email,
  COUNT_IF(gender      IS NULL OR trim(gender)      = '') AS blank_gender,
  COUNT_IF(birth_date  IS NULL OR trim(birth_date)  = '') AS blank_birth_date,
  COUNT_IF(city        IS NULL OR trim(city)        = '') AS blank_city,
  COUNT_IF(region      IS NULL OR trim(region)      = '') AS blank_region,
  COUNT_IF(segment     IS NULL OR trim(segment)     = '') AS blank_segment,
  COUNT_IF(signup_date IS NULL OR trim(signup_date) = '') AS blank_signup_date
FROM bronze_efkleia_urbanmart.customers;

blank_customer_id,blank_first_name,blank_last_name,blank_email,blank_gender,blank_birth_date,blank_city,blank_region,blank_segment,blank_signup_date
0,0,0,15,0,0,0,0,0,0


In [0]:
SELECT 'gender' AS col, gender AS value, COUNT(*) AS n FROM bronze_efkleia_urbanmart.customers GROUP BY gender
UNION ALL
SELECT 'city',    city,    COUNT(*) FROM bronze_efkleia_urbanmart.customers GROUP BY city
UNION ALL
SELECT 'region',  region,  COUNT(*) FROM bronze_efkleia_urbanmart.customers GROUP BY region
UNION ALL
SELECT 'segment', segment, COUNT(*) FROM bronze_efkleia_urbanmart.customers GROUP BY segment
ORDER BY col, value;

col,value,n
city,ATHENS,4
city,Athens,61
city,HERAKLION,3
city,Heraklion,73
city,IOANNINA,7
city,Ioannina,64
city,LARISSA,3
city,Larissa,72
city,PATRAS,5
city,PIRAEUS,2


In [0]:
SELECT customer_id, email, birth_date, signup_date,
       try_to_date(birth_date,  'yyyy-MM-dd') AS birth_parsed,
       try_to_date(signup_date, 'yyyy-MM-dd') AS signup_parsed
FROM bronze_efkleia_urbanmart.customers
WHERE (birth_date  IS NOT NULL AND try_to_date(birth_date,  'yyyy-MM-dd') IS NULL)
   OR (signup_date IS NOT NULL AND try_to_date(signup_date, 'yyyy-MM-dd') IS NULL)
   OR email NOT LIKE '%_@_%._%';

customer_id,email,birth_date,signup_date,birth_parsed,signup_parsed


**Dublicates:** 8 customer IDs appear twice (16 rows, so 8 extra).\
Every one has distinct_versions = 1, so they’re exact copies not conflicting versions. 508 − 8 = 500 real customers\
**Blanks:** 15 customers have an empty email.\
**Text formatting:** 26 of 508 rows (~5%) have city in UPPERCASE with a trailing space (e.g. 'ATHENS '); region, gender and segment are clean and the city–region mapping is consistent, so in silver we standardize with INITCAP(TRIM(city)) \
**Dates and emails:** no bad dates.


**customer_changes**

In [0]:
SELECT
  COUNT(*)                                                       AS total_rows,
  COUNT(DISTINCT customer_id)                                    AS distinct_customers,
  COUNT_IF(customer_id    IS NULL OR trim(customer_id)    = '')  AS blank_customer_id,
  COUNT_IF(new_city       IS NULL OR trim(new_city)       = '')  AS blank_new_city,
  COUNT_IF(new_region     IS NULL OR trim(new_region)     = '')  AS blank_new_region,
  COUNT_IF(new_segment    IS NULL OR trim(new_segment)    = '')  AS blank_new_segment,
  COUNT_IF(effective_date IS NULL OR trim(effective_date) = '')  AS blank_effective_date
FROM bronze_efkleia_urbanmart.customer_changes;

total_rows,distinct_customers,blank_customer_id,blank_new_city,blank_new_region,blank_new_segment,blank_effective_date
25,25,0,0,0,0,0


In [0]:
SELECT MIN(try_to_date(effective_date, 'yyyy-MM-dd'))          AS min_date,
       MAX(try_to_date(effective_date, 'yyyy-MM-dd'))          AS max_date,
       COUNT_IF(try_to_date(effective_date, 'yyyy-MM-dd') IS NULL) AS bad_dates,
       COUNT(DISTINCT new_city)                                AS distinct_cities,
       COUNT(DISTINCT new_segment)                             AS distinct_segments
FROM bronze_efkleia_urbanmart.customer_changes;

min_date,max_date,bad_dates,distinct_cities,distinct_segments
2025-03-03,2025-11-26,0,7,3


In [0]:
SELECT ch.customer_id,
       c.customer_id IS NULL                      AS customer_missing,
       initcap(trim(c.city)) AS current_city,  ch.new_city,
       c.segment             AS current_segment, ch.new_segment,
       initcap(trim(c.city)) = ch.new_city
         AND c.segment = ch.new_segment           AS no_real_change
FROM bronze_efkleia_urbanmart.customer_changes ch
LEFT JOIN (SELECT DISTINCT customer_id, city, segment
           FROM bronze_efkleia_urbanmart.customers) c
  ON ch.customer_id = c.customer_id
ORDER BY no_real_change DESC, ch.customer_id;

customer_id,customer_missing,current_city,new_city,current_segment,new_segment,no_real_change
C0282,false,Heraklion,Heraklion,Loyalty Gold,Loyalty Gold,true
C0027,false,Patras,Heraklion,Retail,Loyalty Gold,false
C0035,false,Larissa,Ioannina,Retail,Retail,false
C0038,false,Piraeus,Athens,Retail,Retail,false
C0095,false,Heraklion,Athens,Retail,Retail,false
C0113,false,Piraeus,Larissa,Retail,Retail,false
C0155,false,Piraeus,Larissa,Retail,Retail,false
C0163,false,Patras,Larissa,Retail,Loyalty Gold,false
C0184,false,Thessaloniki,Thessaloniki,Loyalty Silver,Retail,false
C0252,false,Patras,Athens,Loyalty Silver,Loyalty Gold,false


25 rows, no blanks, all dates valid (2025-03-03 → 2025-11-26), all customers exist. C0282 is a "change" identical to its current values.

**Profiling for products**

In [0]:
SELECT product_id,
       COUNT(*)                         AS row_count,
       COUNT(DISTINCT _source_row_hash) AS distinct_versions
FROM bronze_efkleia_urbanmart.products
GROUP BY product_id
HAVING COUNT(*) > 1;

product_id,row_count,distinct_versions


In [0]:
SELECT
  COUNT_IF(product_id   IS NULL OR trim(product_id)   = '') AS blank_product_id,
  COUNT_IF(product_name IS NULL OR trim(product_name) = '') AS blank_product_name,
  COUNT_IF(category     IS NULL OR trim(category)     = '') AS blank_category,
  COUNT_IF(subcategory  IS NULL OR trim(subcategory)  = '') AS blank_subcategory,
  COUNT_IF(brand        IS NULL OR trim(brand)        = '') AS blank_brand,
  COUNT_IF(unit_cost    IS NULL OR trim(unit_cost)    = '') AS blank_unit_cost,
  COUNT_IF(list_price   IS NULL OR trim(list_price)   = '') AS blank_list_price,
  COUNT_IF(is_active    IS NULL OR trim(is_active)    = '') AS blank_is_active
FROM bronze_efkleia_urbanmart.products;

blank_product_id,blank_product_name,blank_category,blank_subcategory,blank_brand,blank_unit_cost,blank_list_price,blank_is_active
0,0,1,1,1,0,0,0


In [0]:
SELECT 'category' AS col, category AS value, length(category) AS len, COUNT(*) AS n
FROM bronze_efkleia_urbanmart.products GROUP BY category
UNION ALL
SELECT 'brand', brand, length(brand), COUNT(*)
FROM bronze_efkleia_urbanmart.products GROUP BY brand
UNION ALL
SELECT 'is_active', is_active, length(is_active), COUNT(*)
FROM bronze_efkleia_urbanmart.products GROUP BY is_active
ORDER BY col, value;

col,value,len,n
brand,null,null,1
brand,Aegean,6,15
brand,Helios,6,16
brand,Nova,4,14
brand,Olympus,7,14
brand,Zephyr,6,16
category,null,null,1
category,Apparel,7,15
category,Electronics,11,15
category,Grocery,7,15


In [0]:
SELECT product_id, product_name, unit_cost, list_price
FROM bronze_efkleia_urbanmart.products
WHERE try_cast(unit_cost  AS DECIMAL(10,2)) IS NULL      -- not a number
   OR try_cast(list_price AS DECIMAL(10,2)) IS NULL
   OR try_cast(unit_cost  AS DECIMAL(10,2)) <= 0         -- zero or negative
   OR try_cast(list_price AS DECIMAL(10,2)) <= 0
   OR try_cast(list_price AS DECIMAL(10,2))
    < try_cast(unit_cost  AS DECIMAL(10,2));             -- selling below cost

product_id,product_name,unit_cost,list_price
P9999,Unknown Item,0,0


 no duplicates, no invalid prices. One placeholder row P9999 "Unknown Item" (blank category/brand, cost and price 0).

**Profiling stores**

In [0]:
SELECT
  COUNT(*)                                                 AS total_rows,
  COUNT(DISTINCT store_id)                                 AS distinct_store_ids,
  COUNT_IF(store_name IS NULL OR trim(store_name) = '')    AS blank_store_name,
  COUNT_IF(store_type IS NULL OR trim(store_type) = '')    AS blank_store_type,
  COUNT_IF(city       IS NULL OR trim(city)       = '')    AS blank_city,
  COUNT_IF(region     IS NULL OR trim(region)     = '')    AS blank_region,
  COUNT_IF(open_date  IS NULL OR trim(open_date)  = '')    AS blank_open_date,
  COUNT_IF(size_sqm   IS NULL OR trim(size_sqm)   = '')    AS blank_size_sqm
FROM bronze_efkleia_urbanmart.stores;

total_rows,distinct_store_ids,blank_store_name,blank_store_type,blank_city,blank_region,blank_open_date,blank_size_sqm
13,13,0,0,0,0,0,0


In [0]:
SELECT store_id, store_name, store_type, length(store_type) AS type_len,
       city, region, open_date,
       try_to_date(open_date, 'yyyy-MM-dd')   AS open_date_parsed,
       size_sqm,
       try_cast(size_sqm AS INT)              AS size_sqm_parsed
FROM bronze_efkleia_urbanmart.stores
ORDER BY store_id;

store_id,store_name,store_type,type_len,city,region,open_date,open_date_parsed,size_sqm,size_sqm_parsed
S001,UrbanMart Athens 1,High Street,11,Athens,Attica,2018-06-10,2018-06-10,1710,1710
S002,UrbanMart Piraeus 2,Outlet,6,Piraeus,Attica,2019-06-19,2019-06-19,865,865
S003,UrbanMart Thessaloniki 3,Outlet,6,Thessaloniki,Central Macedonia,2016-05-02,2016-05-02,1241,1241
S004,UrbanMart Patras 4,High Street,11,Patras,Western Greece,2022-11-04,2022-11-04,962,962
S005,UrbanMart Heraklion 5,Mall,4,Heraklion,Crete,2017-05-28,2017-05-28,2382,2382
S006,UrbanMart Larissa 6,High Street,11,Larissa,Thessaly,2018-10-18,2018-10-18,513,513
S007,UrbanMart Ioannina 7,Mall,4,Ioannina,Epirus,2019-05-13,2019-05-13,321,321
S008,UrbanMart Athens 8,Mall,4,Athens,Attica,2016-08-18,2016-08-18,2231,2231
S009,UrbanMart Piraeus 9,Outlet,6,Piraeus,Attica,2019-11-19,2019-11-19,661,661
S010,UrbanMart Thessaloniki 10,Mall,4,Thessaloniki,Central Macedonia,2017-12-13,2017-12-13,1208,1208


clean. size_sqm = 0 only for the online store S900, which is valid.

**orders**

In [0]:
SELECT order_id,
       COUNT(*)                         AS row_count,
       COUNT(DISTINCT _source_row_hash) AS distinct_versions
FROM bronze_efkleia_urbanmart.orders
GROUP BY order_id
HAVING COUNT(*) > 1;

order_id,row_count,distinct_versions


In [0]:
SELECT
  COUNT_IF(order_id       IS NULL OR trim(order_id)       = '') AS blank_order_id,
  COUNT_IF(order_ts       IS NULL OR trim(order_ts)       = '') AS blank_order_ts,
  COUNT_IF(customer_id    IS NULL OR trim(customer_id)    = '') AS blank_customer_id,
  COUNT_IF(store_id       IS NULL OR trim(store_id)       = '') AS blank_store_id,
  COUNT_IF(payment_method IS NULL OR trim(payment_method) = '') AS blank_payment_method,
  COUNT_IF(order_status   IS NULL OR trim(order_status)   = '') AS blank_order_status
FROM bronze_efkleia_urbanmart.orders;

blank_order_id,blank_order_ts,blank_customer_id,blank_store_id,blank_payment_method,blank_order_status
0,0,312,0,0,0


In [0]:
SELECT 'payment_method' AS col, payment_method AS value, length(payment_method) AS len, COUNT(*) AS n
FROM bronze_efkleia_urbanmart.orders GROUP BY payment_method
UNION ALL
SELECT 'order_status', order_status, length(order_status), COUNT(*)
FROM bronze_efkleia_urbanmart.orders GROUP BY order_status
UNION ALL
SELECT 'store_id', store_id, length(store_id), COUNT(*)
FROM bronze_efkleia_urbanmart.orders GROUP BY store_id
ORDER BY col, value;

col,value,len,n
order_status,Cancelled,9,302
order_status,Completed,9,7245
order_status,Returned,8,453
payment_method,Card,4,3998
payment_method,Cash,4,2010
payment_method,Mobile Wallet,13,1992
store_id,S001,4,651
store_id,S002,4,610
store_id,S003,4,578
store_id,S004,4,616


In [0]:
SELECT regexp_replace(order_ts, '[0-9]', '9') AS ts_pattern,
       COUNT(*)                               AS n,
       MIN(order_ts)                          AS example
FROM bronze_efkleia_urbanmart.orders
GROUP BY regexp_replace(order_ts, '[0-9]', '9')
ORDER BY n DESC;

ts_pattern,n,example
9999-99-99 99:99:99,7924,2024-01-01 09:47:00
99/99/9999 99:99,76,01/02/2024 12:00


In [0]:
SELECT order_id, order_ts,
       try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss') AS parsed_expected
FROM bronze_efkleia_urbanmart.orders
WHERE try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss') IS NULL
ORDER BY order_id
LIMIT 20;

order_id,order_ts,parsed_expected
O000063,17/02/2025 12:00,null
O000142,25/12/2025 12:00,null
O000381,02/04/2024 12:00,null
O000421,24/10/2025 12:00,null
O000522,04/09/2024 12:00,null
O000711,31/03/2024 12:00,null
O000792,15/01/2024 12:00,null
O000819,28/03/2024 12:00,null
O000841,27/01/2025 12:00,null
O001010,13/05/2024 12:00,null


In [0]:
SELECT MIN(try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')) AS min_ts,
       MAX(try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')) AS max_ts
FROM bronze_efkleia_urbanmart.orders;

min_ts,max_ts
2024-01-01T09:47:00.000Z,2025-12-30T21:52:00.000Z


312 orders with blank customer_id (guest orders). 76 timestamps in dd/MM/yyyy HH:mm format (all at 12:00); the rest are yyyy-MM-dd HH:mm:ss. Date range 2024-01-01 → 2025-12-30.

**Order items**

In [0]:
SELECT order_line_id, COUNT(*) AS row_count
FROM bronze_efkleia_urbanmart.order_items
GROUP BY order_line_id
HAVING COUNT(*) > 1;

order_line_id,row_count


In [0]:
SELECT order_id, product_id, quantity, unit_price, discount_pct,
       COUNT(*)                     AS row_count,
       collect_list(order_line_id)  AS line_ids
FROM bronze_efkleia_urbanmart.order_items
GROUP BY order_id, product_id, quantity, unit_price, discount_pct
HAVING COUNT(*) > 1
ORDER BY order_id;

order_id,product_id,quantity,unit_price,discount_pct,row_count,line_ids
O000151,P0045,4,151.19,0,2,"List(L0000458, L0000460)"
O000451,P0026,3,279.02,0.05,2,"List(L0001349, L0001350)"
O000575,P0004,1,288.91,0,2,"List(L0001735, L0001736)"
O000950,P0020,2,207.01,0.05,2,"List(L0002863, L0002864)"
O000995,P0030,1,219.36,0,2,"List(L0002982, L0002983)"
O001242,P0007,3,63.16,0.2,2,"List(L0003709, L0003710)"
O001248,P0062,2,345.34,0,2,"List(L0003729, L0003730)"
O001252,P0044,3,267.45,0,2,"List(L0003747, L0003748)"
O001313,P0005,4,251.51,0.05,2,"List(L0003931, L0003935)"
O002013,P0073,1,117.92,0,2,"List(L0006039, L0006040)"


In [0]:
SELECT
  COUNT_IF(order_line_id IS NULL OR trim(order_line_id) = '') AS blank_order_line_id,
  COUNT_IF(order_id      IS NULL OR trim(order_id)      = '') AS blank_order_id,
  COUNT_IF(product_id    IS NULL OR trim(product_id)    = '') AS blank_product_id,
  COUNT_IF(quantity      IS NULL OR trim(quantity)      = '') AS blank_quantity,
  COUNT_IF(unit_price    IS NULL OR trim(unit_price)    = '') AS blank_unit_price,
  COUNT_IF(discount_pct  IS NULL OR trim(discount_pct)  = '') AS blank_discount_pct
FROM bronze_efkleia_urbanmart.order_items;

blank_order_line_id,blank_order_id,blank_product_id,blank_quantity,blank_unit_price,blank_discount_pct
0,0,0,0,0,0


In [0]:
SELECT
  COUNT_IF(try_cast(quantity     AS INT)           IS NULL) AS qty_not_integer,
  COUNT_IF(try_cast(quantity     AS INT)           <= 0)    AS qty_zero_or_negative,
  COUNT_IF(try_cast(unit_price   AS DECIMAL(10,2)) IS NULL) AS price_not_number,
  COUNT_IF(try_cast(unit_price   AS DECIMAL(10,2)) <= 0)    AS price_zero_or_negative,
  COUNT_IF(try_cast(discount_pct AS DECIMAL(5,4))  IS NULL) AS discount_not_number,
  COUNT_IF(try_cast(discount_pct AS DECIMAL(5,4)) NOT BETWEEN 0 AND 1) AS discount_out_of_range
FROM bronze_efkleia_urbanmart.order_items;

qty_not_integer,qty_zero_or_negative,price_not_number,price_zero_or_negative,discount_not_number,discount_out_of_range
0,113,0,0,0,0


In [0]:
SELECT 'quantity' AS col, quantity AS value, COUNT(*) AS n
FROM bronze_efkleia_urbanmart.order_items GROUP BY quantity
UNION ALL
SELECT 'discount_pct', discount_pct, COUNT(*)
FROM bronze_efkleia_urbanmart.order_items GROUP BY discount_pct
ORDER BY col, value;

col,value,n
discount_pct,0,12002
discount_pct,0.05,4028
discount_pct,0.1,4046
discount_pct,0.2,4035
quantity,-1,28
quantity,-2,26
quantity,-3,28
quantity,-4,31
quantity,1,6136
quantity,2,5944


 113 negative quantities. 16 pairs with the same product in the same order but different line IDs (legitimate repeat purchases).

**Referential integrity**

In [0]:
SELECT 'orders.customer_id blank (guest?)' AS check_name,
       COUNT(*) AS n
FROM bronze_efkleia_urbanmart.orders
WHERE customer_id IS NULL OR trim(customer_id) = ''
UNION ALL
SELECT 'orders.customer_id not in customers',
       COUNT(*)
FROM bronze_efkleia_urbanmart.orders o
LEFT JOIN (SELECT DISTINCT customer_id FROM bronze_efkleia_urbanmart.customers) c
  ON o.customer_id = c.customer_id
WHERE trim(o.customer_id) <> '' AND c.customer_id IS NULL
UNION ALL
SELECT 'orders.store_id not in stores',
       COUNT(*)
FROM bronze_efkleia_urbanmart.orders o
LEFT JOIN bronze_efkleia_urbanmart.stores s ON o.store_id = s.store_id
WHERE s.store_id IS NULL
UNION ALL
SELECT 'order_items.order_id not in orders',
       COUNT(*)
FROM bronze_efkleia_urbanmart.order_items i
LEFT JOIN bronze_efkleia_urbanmart.orders o ON i.order_id = o.order_id
WHERE o.order_id IS NULL
UNION ALL
SELECT 'order_items.product_id not in products',
       COUNT(*)
FROM bronze_efkleia_urbanmart.order_items i
LEFT JOIN bronze_efkleia_urbanmart.products p ON i.product_id = p.product_id
WHERE p.product_id IS NULL
UNION ALL
SELECT 'order_items using P9999 (Unknown Item)',
       COUNT(*)
FROM bronze_efkleia_urbanmart.order_items
WHERE product_id = 'P9999'
UNION ALL
SELECT 'orders with no order_items',
       COUNT(*)
FROM bronze_efkleia_urbanmart.orders o
LEFT JOIN (SELECT DISTINCT order_id FROM bronze_efkleia_urbanmart.order_items) i
  ON o.order_id = i.order_id
WHERE i.order_id IS NULL;

check_name,n
orders.customer_id blank (guest?),312
orders.customer_id not in customers,0
orders.store_id not in stores,0
order_items.order_id not in orders,1
order_items.product_id not in products,1
order_items using P9999 (Unknown Item),0
orders with no order_items,0


In [0]:
SELECT i.order_line_id, i.order_id, i.product_id, i.quantity, i.unit_price, i.discount_pct,
       o.order_id   IS NULL AS missing_order,
       p.product_id IS NULL AS missing_product
FROM bronze_efkleia_urbanmart.order_items i
LEFT JOIN bronze_efkleia_urbanmart.orders   o ON i.order_id   = o.order_id
LEFT JOIN bronze_efkleia_urbanmart.products p ON i.product_id = p.product_id
WHERE o.order_id IS NULL OR p.product_id IS NULL;

order_line_id,order_id,product_id,quantity,unit_price,discount_pct,missing_order,missing_product
L9999999,O999999,P0001,1,10,0,true,false
L9999998,O000010,PX000,2,15,0,false,true


In [0]:
SELECT i.order_line_id, i.product_id, i.unit_price, p.list_price
FROM bronze_efkleia_urbanmart.order_items i
LEFT JOIN bronze_efkleia_urbanmart.products p ON i.product_id = p.product_id
WHERE try_cast(i.unit_price AS DECIMAL(10,2)) <> try_cast(p.list_price AS DECIMAL(10,2))
   OR p.product_id IS NULL;

order_line_id,product_id,unit_price,list_price
L9999999,P0001,10,266.04
L9999998,PX000,15,null


1 line points to a non-existent order (O999999), 1 line to a non-existent product (PX000). P9999 is not used by any line. Every order has at least one line.

## Data Quality Log

| # | Table | Issue | Rows | Handling rule |
|---|-------|-------|-----:|---------------|
| 1 | customers | Exact duplicate rows | 8 | De-duplicate: keep 1 per `customer_id` |
| 2 | customers | Blank email | 15 | Keep, set to NULL (not needed for analysis) |
| 3 | customers | City uppercase + trailing space | 26 | `initcap(trim(city))` |
| 4 | customer_changes | Change identical to current values | 1 | Ignore in SCD2 (no new version) |
| 5 | products | Placeholder `P9999` Unknown Item | 1 | Exclude from `dim_product`. Unknown = member −1 |
| 6 | stores | `size_sqm = 0` for online store `S900` | 1 | Valid. Keep |
| 7 | orders | Blank `customer_id` (guest orders) | 312 | Keep. Map to customer −1 (Guest) |
| 8 | orders | `order_ts` in `dd/MM/yyyy HH:mm` | 76 | Parse with fallback format. Time is a 12:00 placeholder |
| 9 | order_items | Negative quantity | 113 | Reject → `NEGATIVE_QUANTITY` |
| 10 | order_items | Product not in products (`PX000`) | 1 | Keep. Map to product −1 |
| 11 | order_items | Order not in orders (`O999999`) | 1 | Reject → `ORPHAN_ORDER` |
| 12 | order_items | Same product twice in an order | 32 | Investigated: legitimate (unique line IDs, no gaps). No action |
| 13 | customer_changes (rest), stores, products (rest), orders (dups) | Checked | 0 | No issues found |

###  ❗Stretch 
| # | Table | Issue | Rows | Handling rule |
|---|-------|-------|-----:|---------------|
| 14 | orders / customers | Order date before customer signup_date | 588 lines | Observation, no action. Reason why SCD2 version 1 starts at 1900-01-01 |

# Phase 2 – Dimensional Design (Kimball 4 steps)

**1. Business process:** Retail sales – products sold to customers through 12 physical stores and 1 online store.

**2. Grain of fact_sales:** One row per order line – one product sold in one order, at one store, on one date (`order_line_id`).

**3. Dimensions and business keys**

| Dimension | Business key | Surrogate key | Main attributes |
|---|---|---|---|
| dim_date | date (yyyyMMdd) | date_key (INT) | month, quarter, year, day name, is_weekend |
| dim_customer | customer_id | customer_key | name, email, gender, city, region, segment |
| dim_product | product_id | product_key | name, category, subcategory, brand, unit_cost, list_price |
| dim_store | store_id | store_key | name, store_type, city, region, open_date, size_sqm |

Degenerate dimensions kept in the fact: `order_id`, `order_line_id`, `order_status`, `payment_method`.

**4. Measures**

| Measure | Derivation | Type |
|---|---|---|
| quantity | source | Additive |
| gross_amount | quantity × unit_price | Additive |
| discount_amount | gross_amount × discount_pct | Additive |
| net_amount | gross_amount − discount_amount | Additive |
| cost_amount | quantity × product unit_cost | Additive |
| margin_amount | net_amount − cost_amount | Additive |
| unit_price, discount_pct | source | Non-additive |
| margin % | SUM(margin_amount) / SUM(net_amount), computed in queries | Non-additive |

No semi-additive measures: these are flows (sales), not balances such as stock levels, which cannot be summed over time.

**5. Guest orders and unknown products**
- Guest orders (blank customer_id) are real sales → kept, mapped to `customer_key = -1` ("Guest / Unknown").
- Product not in the catalogue (`PX000`) → kept, mapped to `product_key = -1`. Its cost is unknown (cost_amount = 0).
- Placeholder `P9999 Unknown Item` is not loaded as a product; the -1 member represents it.
- Rule: if a line can be placed in time and space (date + store) it is kept with -1 for the missing part; otherwise (orphan order `O999999`) it goes to `etl_rejects`.

**Design decisions**
- Cancelled and returned orders stay in fact_sales with their `order_status`, so that *bronze lines − rejects = fact rows* holds.
- Revenue, margin and units (Q1–Q3) count `order_status = 'Completed'` only.
- Return rate (Q4) = Returned / (Completed + Returned); cancelled orders are excluded.

ERD: see end of Phase 3. \

❗Stretch: fact_sales also references dim_customer_scd2 through customer_scd2_key (point-in-time).